## Data Preprocessing for ML(Supervised Model - Classification)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)

In [2]:
# Reading Data
df = pd.read_parquet('Clients_LoanLens_clean.parquet')
print(df.shape)
df.head(5)

(347069, 38)


,CODE_GENDER,NAME_FAMILY_STATUS,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,FLAG_OWN_CAR,OWN_CAR_AGE,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_EMP_PHONE,NAME_CONTRACT_STATUS,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE,AMT_ANNUITY,AMT_CREDIT,OTHER_DOCUMENTS,HAS_BUREAU_RECORD,TOTAL_BUREAU_REQ,MONTH_EMPLOYED,AGE_YEARS
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100002,M,Single / not married,0,1,Secondary / secondary special,202500.0,Working,Laborers,Business Entity,N,0.0,Y,House / apartment,0.018801,2,0,0.083037,0.262949,0.139376,1,0,0,0,1,Approved,179055.0,24.0,179055.0,0.0,Consumer loans,New,9251.775,179055.0,0,1,1,21,25
100003,F,Married,0,2,Higher education,270000.0,State servant,Core staff,School,N,0.0,N,House / apartment,0.003541,1,0,0.311267,0.622246,NaN,1,0,0,0,1,Approved,900000.0,12.0,900000.0,0.0,Cash loans,Repeater,98356.995,1035882.0,0,1,0,39,45
100004,M,Single / not married,0,1,Secondary / secondary special,67500.0,Working,Laborers,Government,Y,26.0,Y,House / apartment,0.010032,2,0,NaN,0.555912,0.729567,0,0,0,0,1,Approved,24282.0,4.0,24282.0,4860.0,Consumer loans,New,5357.250,20106.0,0,1,0,7,52
100006,F,Civil marriage,0,2,Secondary / secondary special,135000.0,Working,Laborers,Business Entity,N,0.0,Y,House / apartment,0.008019,2,0,NaN,0.650442,NaN,1,0,0,0,1,Approved,675000.0,48.0,675000.0,0.0,Cash loans,Repeater,24246.000,675000.0,0,0,0,101,52
100007,M,Single / not married,0,1,Secondary / secondary special,121500.0,Working,Core staff,Religion,N,0.0,Y,House / apartment,0.028663,2,0,NaN,0.322738,NaN,0,0,0,1,1,Approved,247500.0,24.0,247500.0,0.0,Cash loans,Repeater,16037.640,274288.5,0,1,0,101,54


In [3]:
df['NAME_CONTRACT_STATUS'].isna().sum()

np.int64(17906)

In [4]:
# Clients with no previous application have no target → not usable for the approval model
df = df.dropna(subset=['NAME_CONTRACT_STATUS'])

print(df.shape)
print(df['NAME_CONTRACT_STATUS'].isna().sum())

(329163, 38)
0


- **No target:** dropped 17,906 clients with no previous application (no Approved / Refused decision). They stay in the clean file for the segmentation notebook.

- Check Target Column (Imbalanced)

In [5]:
print(df['NAME_CONTRACT_STATUS'].value_counts())
print(df['NAME_CONTRACT_STATUS'].value_counts(normalize=True).round(4) * 100)

NAME_CONTRACT_STATUS
Approved    278563
Refused      50600
Name: count, dtype: int64
NAME_CONTRACT_STATUS
Approved    84.63
Refused     15.37
Name: proportion, dtype: float64


- The target is **imbalanced**: 84.6% Approved vs 15.4% Refused. Accuracy alone would be misleading (predicting "Approved" for everyone gives 85%), so we will focus on **Precision, Recall and F1 for the Refused class**.

In [6]:
ratio = df['AMT_CREDIT'] / df['AMT_APPLICATION']
ratio.groupby(df['NAME_CONTRACT_STATUS']).describe().round(2)

,count,mean,std,min,25%,50%,75%,max
NAME_CONTRACT_STATUS,,,,,,,,
Approved,278563.0,1.03,0.13,0.05,1.0,1.00,1.11,2.74
Refused,50600.0,1.09,0.18,0.11,1.0,1.07,1.16,10.00


In [7]:
# Data leakage: AMT_CREDIT is set by the bank with the decision (credit/application ratio differs
# between Approved and Refused), and AMT_ANNUITY is calculated from it and the bank's interest rate
df.drop(columns=['AMT_CREDIT', 'AMT_ANNUITY'], inplace=True)

df.shape

(329163, 36)

- **Data leakage:** `AMT_CREDIT` (approved amount) is set by the bank with the decision: its ratio to the requested amount differs between Approved (median 1.00) and Refused (median 1.07). `AMT_ANNUITY` is calculated from it and the bank's interest rate. Both dropped.

In [8]:
df['NAME_CONTRACT_STATUS'] = df['NAME_CONTRACT_STATUS'].map({'Approved': 0, 'Refused': 1})
df['NAME_CONTRACT_STATUS'].value_counts()

NAME_CONTRACT_STATUS
0    278563
1     50600
Name: count, dtype: int64

In [9]:
print(df[['AMT_APPLICATION', 'AMT_GOODS_PRICE']].corr().round(4))
print(((df['AMT_APPLICATION'] == df['AMT_GOODS_PRICE']).mean() * 100).round(2))

                 AMT_APPLICATION  AMT_GOODS_PRICE
AMT_APPLICATION              1.0              1.0
AMT_GOODS_PRICE              1.0              1.0
99.99


In [10]:
# AMT_GOODS_PRICE = AMT_APPLICATION in 99.99% of applications (correlation = 1.0) → redundant
df.drop(columns=['AMT_GOODS_PRICE'], inplace=True)

df.shape

(329163, 35)

- **AMT_GOODS_PRICE:** equal to `AMT_APPLICATION` in 99.99% of applications (correlation = 1.0) → dropped as redundant.

## Feature Engineering

In [11]:
# How many times the requested amount is the client's income
df['LOAN_TO_INCOME'] = df['AMT_APPLICATION'] / df['AMT_INCOME_TOTAL']
# Share of the amount paid upfront
df['DOWN_PAYMENT_RATIO'] = df['AMT_DOWN_PAYMENT'] / df['AMT_APPLICATION']
# Monthly installment / income (revolving loans have no fixed term → 0)
df['PAYMENT_TO_INCOME'] = (df['AMT_APPLICATION'] / df['CNT_PAYMENT'].replace(0, np.nan)) / df['AMT_INCOME_TOTAL']
df['PAYMENT_TO_INCOME'] = df['PAYMENT_TO_INCOME'].fillna(0)

- **New features:** `LOAN_TO_INCOME`, `DOWN_PAYMENT_RATIO`, `PAYMENT_TO_INCOME` (CNT_PAYMENT = 0 for 33,779 revolving loans → 0). They did not change F1 (tree models already learn these relations from the original columns) but are kept for business interpretation.

### Data Splitting & Preprocessing for ML

In [12]:
x = df.drop(columns=['NAME_CONTRACT_STATUS'])
y = df['NAME_CONTRACT_STATUS']

print(x.shape)
print(y.shape)

(329163, 37)
(329163,)


- Numerical Columns

In [13]:
num_cols = x.select_dtypes(include= 'number').columns
num_cols

Index(['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL', 'OWN_CAR_AGE',
       'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT_W_CITY',
       'LIVE_REGION_NOT_WORK_REGION', 'EXT_SOURCE_1', 'EXT_SOURCE_2',
       'EXT_SOURCE_3', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6',
       'FLAG_DOCUMENT_8', 'FLAG_EMP_PHONE', 'AMT_APPLICATION', 'CNT_PAYMENT',
       'AMT_DOWN_PAYMENT', 'OTHER_DOCUMENTS', 'HAS_BUREAU_RECORD',
       'TOTAL_BUREAU_REQ', 'MONTH_EMPLOYED', 'AGE_YEARS', 'LOAN_TO_INCOME',
       'DOWN_PAYMENT_RATIO', 'PAYMENT_TO_INCOME'],
      dtype='str')

In [14]:
log_cols = ['AMT_INCOME_TOTAL', 'AMT_APPLICATION', 'AMT_DOWN_PAYMENT']

no_scaling_cols = ['FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8',
                   'FLAG_EMP_PHONE', 'LIVE_REGION_NOT_WORK_REGION', 'HAS_BUREAU_RECORD']

scal_cols = ['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'OWN_CAR_AGE', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT_W_CITY',
             'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'CNT_PAYMENT', 'OTHER_DOCUMENTS', 'TOTAL_BUREAU_REQ',
             'MONTH_EMPLOYED', 'AGE_YEARS', 'LOAN_TO_INCOME', 'DOWN_PAYMENT_RATIO', 'PAYMENT_TO_INCOME']

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

num_pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaling', RobustScaler())
])
num_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Imputer', ...), ('Scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account f

In [16]:
from sklearn.preprocessing import FunctionTransformer

log_pipeline = Pipeline(steps=[
    ('Log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('Scaling', RobustScaler())
])
log_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Log', ...), ('Scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<ufunc 'log1p'>
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",'one-to-one'
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"kw_args kw_args: dict, default=NoneDictionary of additional 

- Categorical Columns 

In [17]:
cat_cols = x.select_dtypes(include= 'str').columns
cat_cols

Index(['CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE',
       'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE',
       'NAME_CONTRACT_TYPE', 'NAME_CLIENT_TYPE'],
      dtype='str')

In [18]:
ohe_cols = ['CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE',
            'NAME_CONTRACT_TYPE', 'NAME_CLIENT_TYPE']

ord_cols = ['NAME_EDUCATION_TYPE']

bin_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

In [19]:
from sklearn.preprocessing import OneHotEncoder

ohe_pipeline = Pipeline( [
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(sparse_output= False, drop= 'first', handle_unknown= 'ignore'))
])
ohe_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Imputer', ...), ('OHE', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accoun

In [20]:
x['NAME_EDUCATION_TYPE'].unique()

<ArrowStringArray>
['Secondary / secondary special',              'Higher education',
             'Incomplete higher',               'Lower secondary',
               'Academic degree']
Length: 5, dtype: str

In [21]:
from sklearn.preprocessing import OrdinalEncoder

edu_order = ['Lower secondary', 'Secondary / secondary special', 'Incomplete higher',
             'Higher education', 'Academic degree']

ord_pipeline = Pipeline(steps=[
    ('Ord', OrdinalEncoder(categories=[edu_order]))
])
ord_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Ord', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.","[['Lower secondary', 'Secondary / secondary special', ...]]"
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'error'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",None
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for 

In [22]:
from category_encoders import BinaryEncoder

bin_pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='constant', fill_value='Other')),
    ('BE', BinaryEncoder())
])
bin_pipeline

c:\FKAnaconda\envs\sper\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Example
  self[section] = content


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Imputer', ...), ('BE', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'Other'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account f

In [23]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(transformers=[
    ('Num Pipeline', num_pipeline, scal_cols),
    ('Log Pipeline', log_pipeline, log_cols),
    ('OHE Pipeline', ohe_pipeline, ohe_cols),
    ('Ordinal Pipeline', ord_pipeline, ord_cols),
    ('Binary Pipeline', bin_pipeline, bin_cols)
], remainder='passthrough') # passed through as-is (remainder)
preprocessing

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Num Pipeline', ...), ('Log Pipeline', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transform

### Select The Best Model

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import  RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate

models = [
    ('Logistic Regression', LogisticRegression(random_state=42)),
    #('KNN', KNeighborsClassifier()),
    #('SVM', SVC(random_state= 42)),
    ('Naive Bayes', GaussianNB()),
    ('Decision Tree Classifier', DecisionTreeClassifier(random_state= 42)),
    ('Random Forest Classifier', RandomForestClassifier(random_state= 42)),
    ('XGB Classifier', XGBClassifier(random_state= 42))
]

for model in models:

    model_pipeline = Pipeline(steps= [('Preprocessing', preprocessing), ('Model', model[1])])

    cv_result = cross_validate(model_pipeline, x, y, scoring= ['accuracy', 'precision', 'recall', 'f1'], cv= 5, return_train_score= True, n_jobs= -2)

    print(model[0])

    # Accuracy Score
    print(f"Train Accuracy: {cv_result['train_accuracy'].mean().round(4) * 100:.2f}%")
    print(f"Test Accuracy: {cv_result['test_accuracy'].mean().round(4) * 100:.2f}%")

    # Precision Score
    print(f"Train Precision: {cv_result['train_precision'].mean().round(4) * 100:.2f}%")
    print(f"Test Precision: {cv_result['test_precision'].mean().round(4) * 100:.2f}%")

    # Recall Score
    print(f"Train Recall: {cv_result['train_recall'].mean().round(4) * 100:.2f}%")
    print(f"Test Recall: {cv_result['test_recall'].mean().round(4) * 100:.2f}%")

    # F1 Score
    print(f"Train F1 Score: {cv_result['train_f1'].mean().round(4) * 100:.2f}%")
    print(f"Test F1 Score: {cv_result['test_f1'].mean().round(4) * 100:.2f}%")

    print('-' * 50)

Logistic Regression
Train Accuracy: 85.44%
Test Accuracy: 85.42%
Train Precision: 60.81%
Test Precision: 60.60%
Train Recall: 14.91%
Test Recall: 14.88%
Train F1 Score: 23.95%
Test F1 Score: 23.87%
--------------------------------------------------
Naive Bayes
Train Accuracy: 52.42%
Test Accuracy: 52.42%
Train Precision: 22.89%
Test Precision: 22.88%
Train Recall: 88.46%
Test Recall: 88.38%
Train F1 Score: 36.37%
Test F1 Score: 36.35%
--------------------------------------------------
Decision Tree Classifier
Train Accuracy: 100.00%
Test Accuracy: 78.54%
Train Precision: 100.00%
Test Precision: 31.72%
Train Recall: 100.00%
Test Recall: 34.37%
Train F1 Score: 100.00%
Test F1 Score: 32.99%
--------------------------------------------------
Random Forest Classifier
Train Accuracy: 100.00%
Test Accuracy: 85.64%
Train Precision: 100.00%
Test Precision: 62.60%
Train Recall: 99.98%
Test Recall: 16.40%
Train F1 Score: 99.99%
Test F1 Score: 25.98%
-----------------------------------------------

In [25]:
ratio = (y == 0).sum() / (y == 1).sum()
ratio

np.float64(5.505197628458498)

In [26]:
models = [
    ('Logistic Regression', LogisticRegression(random_state=42, class_weight='balanced')),
    #('KNN', KNeighborsClassifier()),
    #('SVM', SVC(random_state= 42)),
    ('Naive Bayes', GaussianNB()),
    ('Decision Tree Classifier', DecisionTreeClassifier(random_state= 42, class_weight='balanced')),
    ('Random Forest Classifier', RandomForestClassifier(random_state= 42, class_weight='balanced')),
    ('XGB Classifier', XGBClassifier(random_state= 42, scale_pos_weight=ratio))
]

for model in models:

    model_pipeline = Pipeline(steps= [('Preprocessing', preprocessing), ('Model', model[1])])

    cv_result = cross_validate(model_pipeline, x, y, scoring= ['accuracy', 'precision', 'recall', 'f1'], cv= 5, return_train_score= True, n_jobs= -1)

    print(model[0])

    # Accuracy Score
    print(f"Train Accuracy: {cv_result['train_accuracy'].mean().round(4) * 100:.2f}%")
    print(f"Test Accuracy: {cv_result['test_accuracy'].mean().round(4) * 100:.2f}%")

    # Precision Score
    print(f"Train Precision: {cv_result['train_precision'].mean().round(4) * 100:.2f}%")
    print(f"Test Precision: {cv_result['test_precision'].mean().round(4) * 100:.2f}%")

    # Recall Score
    print(f"Train Recall: {cv_result['train_recall'].mean().round(4) * 100:.2f}%")
    print(f"Test Recall: {cv_result['test_recall'].mean().round(4) * 100:.2f}%")

    # F1 Score
    print(f"Train F1 Score: {cv_result['train_f1'].mean().round(4) * 100:.2f}%")
    print(f"Test F1 Score: {cv_result['test_f1'].mean().round(4) * 100:.2f}%")

    print('-' * 50)

Logistic Regression
Train Accuracy: 70.56%
Test Accuracy: 70.52%
Train Precision: 31.43%
Test Precision: 31.37%
Train Recall: 77.47%
Test Recall: 77.32%
Train F1 Score: 44.72%
Test F1 Score: 44.64%
--------------------------------------------------
Naive Bayes
Train Accuracy: 52.42%
Test Accuracy: 52.42%
Train Precision: 22.89%
Test Precision: 22.88%
Train Recall: 88.46%
Test Recall: 88.38%
Train F1 Score: 36.37%
Test F1 Score: 36.35%
--------------------------------------------------
Decision Tree Classifier
Train Accuracy: 100.00%
Test Accuracy: 79.05%
Train Precision: 100.00%
Test Precision: 31.59%
Train Recall: 100.00%
Test Recall: 31.16%
Train F1 Score: 100.00%
Test F1 Score: 31.37%
--------------------------------------------------
Random Forest Classifier
Train Accuracy: 99.94%
Test Accuracy: 83.90%
Train Precision: 99.62%
Test Precision: 47.29%
Train Recall: 100.00%
Test Recall: 41.31%
Train F1 Score: 99.81%
Test F1 Score: 44.10%
------------------------------------------------

- **Without class weights**, most models predicted "Approved" for almost everyone (Recall 15–22%, below the 0.3 requirement), since only 15.4% of applications are refused.
- **With `class_weight='balanced'` / `scale_pos_weight`**, Logistic Regression and XGBoost passed (Precision ≥ 0.31, Recall ≥ 0.75). **XGBoost has the best F1 (46.7%)** with little overfitting.

### Hyperparameter Tuning

- Logistic Regression

In [27]:
from sklearn.model_selection import GridSearchCV

lr_pipeline = Pipeline(steps=[('Preprocessing', preprocessing),
                              ('Model', LogisticRegression(random_state=42, class_weight='balanced', max_iter= 1000))])

lr_param_grid = {
    'Model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'Model__penalty': ['l2']
}

grid_search_lr = GridSearchCV(estimator=lr_pipeline, param_grid=lr_param_grid,
                              scoring=['f1', 'precision', 'recall'], refit='f1',
                              cv=5, n_jobs=-1, return_train_score=True)

grid_search_lr.fit(x, y)

c:\FKAnaconda\envs\sper\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'Model__C': [0.001, 0.01, ...], 'Model__penalty': ['l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","['f1', 'precision', ...]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, defa

In [28]:
grid_search_lr.best_params_

{'Model__C': 0.1, 'Model__penalty': 'l2'}

In [29]:
grid_search_lr.best_score_

np.float64(0.44659664035243934)

- XGBoost

In [30]:
xgb_pipeline = Pipeline(steps=[('Preprocessing', preprocessing),
                               ('Model', XGBClassifier(random_state=42, scale_pos_weight=ratio))])

xgb_param_grid = {
    'Model__n_estimators': [100, 200],
    'Model__max_depth': [3, 5, 7],
    'Model__learning_rate': [0.03, 0.1]
}

grid_search_xgb = GridSearchCV(estimator=xgb_pipeline, param_grid=xgb_param_grid,
                               scoring=['f1', 'precision', 'recall'], refit='f1',
                               cv=5, n_jobs=-2, return_train_score=True)

grid_search_xgb.fit(x, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'Model__learning_rate': [0.03, 0.1], 'Model__max_depth': [3, 5, ...], 'Model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","['f1', 'precision', ...]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"

In [31]:
grid_search_xgb.best_params_

{'Model__learning_rate': 0.1,
 'Model__max_depth': 7,
 'Model__n_estimators': 200}

In [32]:
grid_search_xgb.best_score_

np.float64(0.47000863310838853)

In [33]:
def show_best(grid):
    best = grid.best_index_
    res = grid.cv_results_
    print(grid.best_params_)
    for m in ['f1', 'precision', 'recall']:
        print(f"{m:10} Train: {res['mean_train_' + m][best]*100:.2f}%   Test: {res['mean_test_' + m][best]*100:.2f}%")
    print('-' * 50)

show_best(grid_search_lr)
show_best(grid_search_xgb)

{'Model__C': 0.1, 'Model__penalty': 'l2'}
f1         Train: 44.70%   Test: 44.66%
precision  Train: 31.41%   Test: 31.38%
recall     Train: 77.49%   Test: 77.42%
--------------------------------------------------
{'Model__learning_rate': 0.1, 'Model__max_depth': 7, 'Model__n_estimators': 200}
f1         Train: 51.53%   Test: 47.00%
precision  Train: 37.51%   Test: 34.24%
recall     Train: 82.30%   Test: 74.91%
--------------------------------------------------


- **Tuning (GridSearchCV, cv = 5, scoring = F1):** Logistic Regression F1 stayed ~44.7% for all `C` values. XGBoost best `learning_rate = 0.1`, `max_depth = 7`, `n_estimators = 200` → **F1 = 47.00%, Precision = 34.24%, Recall = 74.91%**, with small overfitting → **XGBoost selected**.

## Create Final Model

In [34]:
xgboost_pipeline = Pipeline(steps=[('Preprocessing', preprocessing),
                                   ('Model', XGBClassifier(random_state=42, scale_pos_weight=ratio,
                                                           learning_rate= 0.1,
                                                           max_depth= 7,
                                                           n_estimators= 200))])
xgboost_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Preprocessing', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Num Pipeline', ...), ('Log Pipeline', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_thre

### Fit Final model to data

In [35]:
xgboost_pipeline.fit(x, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Preprocessing', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](37,)","['CODE_GENDER','NAME_FAMILY_STATUS','CNT_CHILDREN',...,'LOAN_TO_INCOME', 'DOWN_PAYMENT_RATIO','PAYMENT_TO_INCOME']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,37
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Num Pipeline', ...), ('Log Pipeline', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (def

### Save Model

In [36]:
import joblib

# Save final model + input columns (for the Streamlit app)
joblib.dump(xgboost_pipeline, 'XGBoost_Approval_Pipeline.pkl')
joblib.dump(x.columns.tolist(), 'Inputs.pkl')

['Inputs.pkl']

### Test Model

In [37]:
model = joblib.load('XGBoost_Approval_Pipeline.pkl')

sample = x.sample(5, random_state=42)
print(model.predict(sample))
print(model.predict_proba(sample)[:, 1].round(3))
print(y.loc[sample.index].values)

[0 0 1 0 0]
[0.122 0.344 0.641 0.139 0.029]
[0 0 1 0 0]


## Feature Importance

In [38]:
# Embedded feature selection: XGBoost importance of each feature
feature_names = xgboost_pipeline.named_steps['Preprocessing'].get_feature_names_out()
importances = xgboost_pipeline.named_steps['Model'].feature_importances_

fi = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi['Feature'] = fi['Feature'].str.split('__').str[-1]      # remove pipeline prefix
fi = fi.sort_values('Importance', ascending=False)

fi.head(20)

,Feature,Importance
37,NAME_CONTRACT_TYPE_Consumer loans,0.676125
40,NAME_CLIENT_TYPE_Repeater,0.089950
8,CNT_PAYMENT,0.022658
7,EXT_SOURCE_3,0.011574
39,NAME_CLIENT_TYPE_Refreshed,0.010796
57,FLAG_DOCUMENT_8,0.007965
56,FLAG_DOCUMENT_6,0.007960
55,FLAG_DOCUMENT_5,0.007893
20,NAME_FAMILY_STATUS_Married,0.007554
17,AMT_APPLICATION,0.005900


In [39]:
px.bar(fi.head(20), x='Importance', y='Feature', orientation='h',
       title='Top 20 Features (XGBoost)').update_yaxes(autorange='reversed')

- **Embedded feature selection (XGBoost importance):** `NAME_CONTRACT_TYPE` is by far the most important feature: consumer loans are refused only 4.9% of the time (secured by the goods) vs ~27% for cash and revolving loans.
- `NAME_CLIENT_TYPE` comes second: new clients are refused 1.9% vs 20.9% for repeaters.
- Then `CNT_PAYMENT`, `EXT_SOURCE_3` and the documents. The new features `PAYMENT_TO_INCOME` and `DOWN_PAYMENT_RATIO` are in the top 20.